In [ ]:
#imports

import pandas as pd 
import numpy as np 
import datetime
import matplotlib.pyplot as plt

In [ ]:
# Loading Datasets

stock = pd.read_csv("HistoricalData_GOOGLE.csv")
treasury = pd.read_csv("HistoricalData_10YrTreasury.csv")

In [ ]:
#Standardizing Alphabet(Google) Stock Price Dataset

stock["Date"] = pd.to_datetime(stock["Date"])
stock["Close/Last"] = stock["Close/Last"].replace(r"[\$,]", "", regex=True).astype(float)
stock = stock.drop(columns = ["Volume", "Open", "High", "Low"])
stock = stock.rename(columns = {"Close/Last": "Price"})
print(stock)

In [ ]:
# Identifying variable formats

stock.dtypes

In [ ]:
#Checking for missing values

stock.isnull().sum()

In [ ]:
# Identifying variable formats

treasury.dtypes

In [ ]:
#Checking for missing values

treasury.isnull().sum()

In [ ]:
#Standardizing US 10-year Treasury Yield Dataset

treasury = treasury.rename(columns = {"observation_date": "Date", "DGS10": "Treasury Rate"})
treasury["Date"] = pd.to_datetime(treasury["Date"])

In [ ]:
# Merging both datasets

merged_dataset = pd.merge(stock, treasury, on="Date", how="inner")

In [ ]:
#Filling missing values

merged_dataset["Treasury Rate"] = merged_dataset["Treasury Rate"].ffill()
merged_dataset["Return"] = merged_dataset["Price"].pct_change()
print(merged_dataset)

In [ ]:
#Checking for missing values

merged_dataset.isnull().sum()

In [ ]:
#Calculating Daily Price Change (Value in Percent)

merged_dataset["Return"] = merged_dataset["Price"].pct_change() *100
merged_dataset["Return"]

In [ ]:
#  Benchmark Analysis

avg_daily_return = merged_dataset["Return"].mean()
annual_return = avg_daily_return * 252

daily_volatility = merged_dataset["Return"].std()
annual_volatility = daily_volatility * np.sqrt(252)

print("Average Daily Return:", avg_daily_return)
print("Annualized Return:", annual_return)
print("Daily Volatility:", daily_volatility)
print("Annualized Volatility:", annual_volatility)

In [ ]:
# Risk to Reward Ratio

risk_to_reward = annual_volatility / annual_return

print("Risk to Reward Ratio:", risk_to_reward)

In [ ]:
# Risk Premium

risk_premium = annual_return - ((merged_dataset["Treasury Rate"] / 100).mean())

print("Risk Premium:", risk_premium)

In [ ]:
# Correlation

correlation = merged_dataset["Return"].corr(
    merged_dataset["Treasury Rate"].diff()
)

print("Correlation between Alphabet Returns and Treasury Yield Changes:")
print(correlation)

In [ ]:
# DAILY STOCK RETURNS OF ALPHABET

plt.figure(figsize=(8,5))

plt.plot(
    merged_dataset["Date"],
    merged_dataset["Return"]
)

plt.title("Alphabet Daily Stock Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.show()

In [ ]:
# TREASURY YIELD OVER TIME

plt.figure(figsize=(8,5))

plt.plot(
    merged_dataset["Date"],
    merged_dataset["Treasury Rate"]
)

plt.title("US 10-Year Treasury Yield")
plt.xlabel("Date")
plt.ylabel("Yield (%)")
plt.show()

In [ ]:
# RETURNS vs RATE CHANGES

plt.figure(figsize=(8,5))

plt.scatter(
    merged_dataset["Treasury Rate"].diff(),
    merged_dataset["Return"]
)

plt.title("Alphabet Returns vs Treasury Yield Changes")
plt.xlabel("Treasury Yield Change")
plt.ylabel("Alphabet Daily Return")
plt.show()

In [ ]:
# RETURN vs RISK vs PREMIUM

metrics = [
    annual_return,
    annual_volatility,
    risk_premium
]

labels = [
    "Annualized Return",
    "Annualized Volatility",
    "Risk Premium"
]

plt.figure(figsize=(8,5))
plt.bar(labels, metrics)
plt.title("Return vs Volatility vs Risk Premium")
plt.ylabel("Value")
plt.show()